In [1]:
from pathlib import Path
import re
from bisect import bisect_right
from liberty_parser import LP

# Работает при запуске из корня проекта или каталога lab2.
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "PDK/IHP_Open_PDK").is_dir())
liberty_file = project_root / "PDK/IHP_Open_PDK/sg13g2_stdcell/lib/sg13g2_stdcell_typ_1p20V_25C.lib"
library = LP(str(liberty_file))
cell = "sg13g2_inv_1"
# Единицы библиотеки: ёмкости — пФ, время — нс.


In [2]:
def get_input_capacitance(cell: str, pin: str) -> float:
    return float(library.navigate_single(
        "library", f"cell:{cell}", f"pin:{pin}", "capacitance").get_val())


def numbers(text):
    return [float(x) for x in re.findall(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?", text)]


def get_table(cell, pin, table_name):
    def read(attribute):
        return library.navigate_single(
            "library", f"cell:{cell}", f"pin:{pin}",
            "timing", table_name, attribute).get_val()

    transitions = numbers(read("index_1"))
    loads = numbers(read("index_2"))
    values = numbers(read("values"))
    if len(values) != len(transitions) * len(loads):
        raise ValueError(f"Неверный размер таблицы {table_name}")
    rows = [values[i:i + len(loads)]
            for i in range(0, len(values), len(loads))]
    return transitions, loads, rows


def interval(axis, value):
    if not axis[0] <= value <= axis[-1]:
        raise ValueError(f"{value} вне диапазона [{axis[0]}, {axis[-1]}]")
    return min(max(bisect_right(axis, value) - 1, 0), len(axis) - 2)


def interpolate(table, transition, load):
    transitions, loads, rows = table
    i = interval(transitions, transition)
    j = interval(loads, load)
    a = (transition - transitions[i]) / (transitions[i + 1] - transitions[i])
    b = (load - loads[j]) / (loads[j + 1] - loads[j])
    f1 = rows[i][j] + b * (rows[i][j + 1] - rows[i][j])
    f2 = rows[i + 1][j] + b * (rows[i + 1][j + 1] - rows[i + 1][j])
    return f1 + a * (f2 - f1)


def calculate_inverter(name, input_edge, input_transition, load):
    output_edge = "fall" if input_edge == "rise" else "rise"
    delay = interpolate(get_table(cell, "Y", f"cell_{output_edge}"),
                        input_transition, load)
    output_transition = interpolate(
        get_table(cell, "Y", f"{output_edge}_transition"), input_transition, load)
    return dict(name=name, input_edge=input_edge, output_edge=output_edge,
                input_transition=input_transition, load=load,
                delay=delay, output_transition=output_transition)


In [3]:
cin = get_input_capacitance(cell, "A")
# Ёмкости и задержки проводов по условию равны нулю.
inv1 = calculate_inverter("INV1", "rise", 0.08, 2 * cin)
inv2 = calculate_inverter("INV2", inv1["output_edge"], inv1["output_transition"], 0.03)
inv3 = calculate_inverter("INV3", inv1["output_edge"], inv1["output_transition"], 0.05)

print(f"Библиотека: {liberty_file.name}")
print(f"Входная ёмкость: {cin:.7f} пФ; нагрузка INV1: {2 * cin:.7f} пФ")
print("\nЭлемент  Вход→выход  Входной tr, нс  Нагрузка, пФ  Задержка, нс  Выходной tr, нс")
for result in (inv1, inv2, inv3):
    edge = f"{result['input_edge']}→{result['output_edge']}"
    print(f"{result['name']:<8} {edge:<11} {result['input_transition']:.9f}   "
          f"{result['load']:.7f}     {result['delay']:.9f}   "
          f"{result['output_transition']:.9f}")

print("\nПуть                  Задержка, нс    Transition на выходе, нс")
for endpoint, last in (("Y", inv2), ("Z", inv3)):
    total_delay = inv1["delay"] + last["delay"]
    print(f"X → INV1 → {last['name']} → {endpoint}: {total_delay:.9f}     "
          f"{last['output_transition']:.9f}")
# Transition вдоль пути не складываются: каждый следующий элемент
# получает выходной transition предыдущего как свой входной.


Библиотека: sg13g2_stdcell_typ_1p20V_25C.lib
Входная ёмкость: 0.0028869 пФ; нагрузка INV1: 0.0057738 пФ

Элемент  Вход→выход  Входной tr, нс  Нагрузка, пФ  Задержка, нс  Выходной tr, нс
INV1     rise→fall   0.080000000   0.0057738     0.050865253   0.035647949
INV2     fall→rise   0.035647949   0.0300000     0.112475804   0.128958930
INV3     fall→rise   0.035647949   0.0500000     0.168038322   0.208281255

Путь                  Задержка, нс    Transition на выходе, нс
X → INV1 → INV2 → Y: 0.163341057     0.128958930
X → INV1 → INV3 → Z: 0.218903575     0.208281255
